# Day 5 · Concept 1 of 5: The Timeout

**Big idea:** never wait forever for something you do not control.
*About 10 minutes. You can follow along without writing any code.*

---

### The idea in plain words
Imagine you phone your card company to ask for your balance, and the line just stays silent. You would not hold the phone for three hours. You would decide in advance, "I will wait 30 seconds, then hang up."

That "hang up after a while" rule is a **timeout**.

### What went wrong in L2
In L2, our SupportDesk agent called its tool and simply **hoped it would answer**. If the tool froze, the customer's request froze too, and nobody was told why.

### What we do now (L3)
Every call to something outside our control gets a time limit. When the limit is reached, the agent stops waiting and gives the customer a clear, polite answer.

| Before (L2) | Now (L3) |
| --- | --- |
| Wait for the tool as long as it takes | Wait at most a fixed number of seconds |
| Customer sees nothing | Customer sees a clear message |

### Step 1: Connect to the AI model
The agent uses the same local model (Ollama) as in earlier days to write friendly replies.

In [ ]:
# Step 0 - Connect to the AI model (Ollama). Run this cell first.
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)
try:
    print("Model says:", llm.invoke("Reply with just the word: ready").content)
except Exception as problem:
    print("The model is not reachable yet.")
    print("Fix: open a terminal, run   ollama signin   then restart this notebook.")
    print("Technical detail:", type(problem).__name__)

### Step 2: The pretend card company
We cannot break a real bank, so we use a pretend one. It answers correctly, but we can tell it **how long to take**.

In [ ]:
import time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

from concurrent.futures import ThreadPoolExecutor, TimeoutError as WaitedTooLong

def card_company_balance(user, seconds_to_answer):
    """The pretend card company. It takes as long as we tell it to."""
    time.sleep(seconds_to_answer)
    return "1,250 INR"

TIMEOUT_SECONDS = 1.0   # <- the hang-up rule: wait at most this long
pool = ThreadPoolExecutor(max_workers=4)

### Step 3: A tiny LangGraph agent with a timeout
A LangGraph agent is a set of **boxes (nodes)** joined by **arrows**. Ours has just two boxes:

```
START  ->  [ call_tool ]  ->  [ write_reply ]  ->  END
```

* **call_tool** phones the card company, but only waits `TIMEOUT_SECONDS`.
* **write_reply** tells the customer the result. If the call timed out it uses a fixed polite message. Otherwise the AI model writes a friendly sentence.

In [ ]:
class State(TypedDict):          # the "case file" that travels through the boxes
    user: str
    seconds_to_answer: float
    balance: str
    error: str
    reply: str

def call_tool(state):
    future = pool.submit(card_company_balance, state["user"], state["seconds_to_answer"])
    try:
        balance = future.result(timeout=TIMEOUT_SECONDS)     # <- the hang-up rule
        return {"balance": balance, "error": ""}
    except WaitedTooLong:
        return {"balance": "", "error": "timeout"}

def write_reply(state):
    if state["error"]:
        return {"reply": "Sorry, our bank system is slow right now. Please try again in a minute."}
    fact = f"The customer's balance is {state['balance']}."
    return {"reply": llm.invoke("You are a polite bank assistant. Say this to the customer in one short sentence. " + fact).content}

builder = StateGraph(State)
builder.add_node("call_tool", call_tool)
builder.add_node("write_reply", write_reply)
builder.add_edge(START, "call_tool")
builder.add_edge("call_tool", "write_reply")
builder.add_edge("write_reply", END)
agent = builder.compile()

def ask(seconds_to_answer):
    started = time.time()
    result = agent.invoke({"user": "cust-1", "seconds_to_answer": seconds_to_answer,
                           "balance": "", "error": "", "reply": ""})
    print(f"Card company needs {seconds_to_answer}s, we wait at most {TIMEOUT_SECONDS}s")
    print(f"  Customer waited : {time.time() - started:.1f} seconds")
    print(f"  Customer sees   : {result['reply']}")

### Step 4: Run it two ways
First a **fast** card company, then a **frozen** one (it would take 3 seconds). Watch how long the customer waits.

In [ ]:
ask(0.2)     # a healthy card company
print()
ask(3.0)     # a frozen card company: we give up after 1 second

**What you should see:** the first answer arrives quickly and sounds friendly (the model wrote it). The second one gives up after about **1 second** with the polite fixed message. We never waited the full 3 seconds.

### Your turn: predict first, then run
Change the hang-up rule to **5 seconds** and ask the frozen card company (3 seconds) again.

**Predict:** will the customer get a balance, or the "slow" message? How long will they wait?

In [ ]:
TIMEOUT_SECONDS = 5.0
ask(3.0)
TIMEOUT_SECONDS = 1.0     # put the rule back

**Answer:** the customer waits the full 3 seconds and gets the balance. The timeout is a *choice*: too short and you give up on healthy calls, too long and customers wait. Teams pick it from how long the tool normally takes.

---

### Remember
* A timeout turns a **silent hang** into a **clear answer**.
* The agent decides the limit, not the tool.

> **Technical note (optional):** Python cannot forcibly stop the slow call, so it finishes in the background. Production systems also set the timeout inside the network client so the call itself is cancelled. Day 19 uses `asyncio.wait_for` with `MCP_TIMEOUT_S`.

**Next notebook:** the card company said "try again", so what do we do? Retrying, politely.